In [ ]:
import gradio as gr
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# --- Helper: Histogram Plot ---
def plot_histogram(img):
    if img is None: return None
    plt.figure(figsize=(3, 2))
    plt.hist(img.flatten(), bins=64, color='gray', alpha=0.7)
    plt.title("Intensity Distribution", fontsize=8)
    plt.xticks(fontsize=6); plt.yticks(fontsize=6)
    plt.tight_layout(); plt.savefig("hist_temp.png"); plt.close()
    return "hist_temp.png"

# --- TAB 1: Restoration & Enhancement ---
def dynamic_restoration(src, n_type, n_lvl, p_method, gamma, f_type, k_size, d0):
    if src is None:
       return None, None, None, "Waiting for Image..."
    src = cv2.resize(src, (350, 350))
    gray_orig = cv2.cvtColor(src, cv2.COLOR_RGB2GRAY) if len(src.shape) == 3 else src

    # 1. Optional Degradation (Noise Injection)
    noisy = gray_orig.copy()
    if n_type == "Salt & Pepper":
        total = gray_orig.size
        num = int(n_lvl * total)
        # Salt
        coords = [np.random.randint(0, i, num) for i in gray_orig.shape]
        noisy[coords[0], coords[1]] = 255
        # Pepper
        coords = [np.random.randint(0, i, num) for i in gray_orig.shape]
        noisy[coords[0], coords[1]] = 0
        pass

        #Gaussian
    elif n_type == "Gaussian":
        gauss = np.random.normal(0, n_lvl * 100, gray_orig.shape)
        noisy = np.clip(gray_orig.astype(np.float32) + gauss, 0, 255).astype(np.uint8)
        pass

        #Periodic
    elif n_type == "Periodic":
        rows, cols = gray_orig.shape
        x = np.arange(cols)
        y = np.arange(rows)
        X, Y = np.meshgrid(x, y)

        amplitude = n_lvl * 255
        frequency = 10

        periodic_noise = amplitude * np.sin(2 * np.pi * frequency * X / cols)
        noisy = gray_orig.astype(np.float32) + periodic_noise
        noisy = np.clip(noisy, 0, 255).astype(np.uint8)

        #Gamma
    elif n_type == "Gamma Noise":
        shape = 2.0

        gamma_noise = np.random.gamma(shape, 1.0, gray_orig.shape).astype(np.float32)
        gamma_noise = gamma_noise - np.mean(gamma_noise)
        gamma_noise = gamma_noise / (np.std(gamma_noise) + 1e-8)

        amplitude = n_lvl * 255

        noisy = gray_orig.astype(np.float32) + amplitude * gamma_noise
        noisy = np.clip(noisy, 0, 255).astype(np.uint8)

    # 2. Intensity Transformations (Enhancement)
    work = noisy.astype(np.float32)
    if p_method == "Negative":
        work = 255 - work # Example implemented
    elif p_method == "Log":
        #  Implement Log Transformation here
        c = 255 / np.log(1 + np.max(work))
        work = c * np.log(1 + work)
        pass
    elif p_method == "Gamma":
        #Implement Gamma Correction here
        work = np.power(work / 255.0, gamma) * 255.0
        pass
    elif p_method == "Histogram Equalization":
        work = cv2.equalizeHist(work.astype(np.uint8))

    processed = np.clip(work, 0, 255).astype(np.uint8)

    # 3. Filtering (Restoration)
    restored = processed.copy()
    if f_type != "None":
        k = int(k_size) | 1
        if f_type == "Mean":
            restored = cv2.blur(processed, (k, k)) # Example implemented
        elif f_type == "Median":
              restored = cv2.medianBlur(processed, k)
        elif f_type == "Laplacian":
            #Implement Laplacian Sharpening here
            lap = cv2.Laplacian(processed, cv2.CV_64F)
            restored = np.clip(processed - lap, 0, 255).astype(np.uint8)
        elif f_type in ["Low Pass", "High Pass"]:
            #Implement Frequency Domain Filtering (FFT) here
            F = np.fft.fft2(processed)
            F_shift = np.fft.fftshift(F)
            rows, cols = processed.shape
            crow, ccol = rows // 2, cols // 2
            U, V = np.ogrid[:rows, :cols]
            D = np.sqrt((U - crow)**2 + (V - ccol)**2)

            H = (D <= d0).astype(np.float32)

            if f_type == "High Pass":
                H = 1 - H

            G = F_shift * H

            restored = np.real(np.fft.ifft2(np.fft.ifftshift(G)))
            restored = np.clip(restored, 0, 255).astype(np.uint8)
        elif f_type == "Notch":
            F = np.fft.fft2(processed)
            F_shift = np.fft.fftshift(F)

            rows, cols = processed.shape
            crow, ccol = rows // 2, cols // 2

            U, V = np.ogrid[:rows, :cols]

            notch_radius = max(1, min(int(d0), 20))
            offset = 10

            D1 = np.sqrt((U - crow)**2 + (V - (ccol + offset))**2)
            D2 = np.sqrt((U - crow)**2 + (V - (ccol - offset))**2)

            H = np.ones((rows, cols), dtype=np.float32)
            H[D1 <= notch_radius] = 0
            H[D2 <= notch_radius] = 0

            G = F_shift * H
            restored = np.real(np.fft.ifft2(np.fft.ifftshift(G)))
            restored = np.clip(restored, 0, 255).astype(np.uint8)

    # 4. Quantitative Metrics
    mse_val = np.mean((gray_orig.astype(np.float32) - restored.astype(np.float32))**2)
    psnr = 100 if mse_val == 0 else 20 * np.log10(255.0 / np.sqrt(mse_val))

    return restored, cv2.absdiff(gray_orig, restored), plot_histogram(restored), f"MSE: {mse_val:.2f} | PSNR: {psnr:.2f} dB"

# --- TAB 2: Segmentation & Morphology ---
def dynamic_segmentation(img, seg_meth, threshold, morph_op, se_shape, se_size, class_label):
    if img is None: return [None]*3
    img = cv2.resize(img, (350, 350))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if len(img.shape) == 3 else img

    # 1. Segmentation
        # 1. Segmentation
    if seg_meth == "Global":
        _, binary = cv2.threshold(
            gray, threshold, 255, cv2.THRESH_BINARY_INV
        )

    elif seg_meth == "Adaptive":
        binary = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV,
            11, 2
        )

    elif seg_meth == "Otsu":
        _, binary = cv2.threshold(
            gray, 0, 255,
            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

    # 2. Morphology
    s_size = int(se_size)
    # Implement Structuring Element (SE) logic for Square, Cross, and Disk
    if se_shape == "Square":
        se = cv2.getStructuringElement(cv2.MORPH_RECT, (s_size, s_size))
    elif se_shape == "Cross":
        se = cv2.getStructuringElement(cv2.MORPH_CROSS, (s_size, s_size))
    elif se_shape == "Disk":
        se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (s_size, s_size))

    m_out = binary.copy()
    if morph_op == "Erosion":
        m_out = cv2.erode(binary, se) # Example implemented
    #  the selected Morphological operation here
    elif morph_op == "Boundary Extraction":
        m_out = cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, se)
    elif morph_op == "Dilation":
        m_out = cv2.dilate(binary, se)
    elif morph_op == "Opening":
        m_out = cv2.morphologyEx(binary, cv2.MORPH_OPEN, se)
    elif morph_op == "Closing":
        m_out = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, se)
    elif morph_op == "Boundary Extraction":
        m_out = cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, se)

    # 3. Feature Extraction
    contours, _ = cv2.findContours(m_out, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    _, hier = cv2.findContours(m_out, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    euler_number = 0
    if hier is not None:
        for i in range(len(hier[0])):
            if hier[0][i][3] == -1:
                euler_number += 1
            else:
                euler_number -= 1
    feats = []
    for i, c in enumerate(contours):
        area = cv2.contourArea(c)
        if area < 50:
            continue

        perim = cv2.arcLength(c, True)
        #  Implement Circularity/Compactness formula here
        circ = (4 * np.pi * area) / (perim ** 2)
        compactness = (perim ** 2) / (4 * np.pi * area)

        #  Implement Hu Moment 1(Hu_M1) formula here
        M = cv2.moments(c)
        hu = cv2.HuMoments(M).flatten()

        feats.append({
            "Area": round(area,1),
            "Perimeter": round(perim,1),
            "Circularity": round(circ,3),
            "Compactness": round(compactness, 3),
            "Hu_M1": hu[0],
            "Euler": euler_number,
            "Class": class_label})

    df = pd.DataFrame(feats)
    df.to_csv("current_sample.csv", index=False)
    return m_out, df, "current_sample.csv"

# --- TAB 3: Batch Analysis (Logic remains for evaluation) ---
def run_classification_analysis(train_file, test_file):
    try:
        df_tr = pd.read_csv(train_file.name) if train_file.name.endswith('.csv') else pd.read_excel(train_file.name)
        df_ts = pd.read_csv(test_file.name) if test_file.name.endswith('.csv') else pd.read_excel(test_file.name)

        df_tr.columns = df_tr.columns.str.strip()
        df_ts.columns = df_ts.columns.str.strip()

        cols = ['Area', 'Perimeter', 'Circularity', 'Compactness', 'Hu_M1', 'Euler']

        missing_train = [c for c in cols if c not in df_tr.columns]
        missing_test = [c for c in cols if c not in df_ts.columns]

        if missing_train or missing_test:
            return None, None, f"Missing columns. Train: {missing_train}, Test: {missing_test}"

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(df_tr[cols].values)
        X_test_scaled = scaler.transform(df_ts[cols].values)

        pca = PCA(n_components=2)
        X_train_pca = pca.fit_transform(X_train_scaled)
        X_test_pca = pca.transform(X_test_scaled)

        # Minimum Distance Classification
        preds = []
        for vec in X_test_pca:
            dists = [np.linalg.norm(vec - tr_vec) for tr_vec in X_train_pca]
            preds.append(df_tr.iloc[np.argmin(dists)]['Class'])

        df_ts['Prediction'] = preds

        # PCA Plot
        plt.figure(figsize=(12, 8))

        for label in df_tr['Class'].unique():
            row_idx = df_tr[df_tr['Class'] == label].index
            plt.scatter(
                X_train_pca[row_idx, 0],
                X_train_pca[row_idx, 1],
                label=f"Prototype: {label}",
                s=300,
                edgecolors='k',
                alpha=0.6
            )

        for j in range(len(df_ts)):
            plt.scatter(X_test_pca[j, 0], X_test_pca[j, 1], s=250, marker='X', c='red')
            plt.text(
                X_test_pca[j, 0] + 0.05,
                X_test_pca[j, 1] + 0.05,
                f"Test {j+1}: {preds[j]}",
                fontsize=10,
                fontweight='bold',
                color='red'
            )

        plt.title("PCA Feature Space: Training Prototypes vs. Test Samples")
        plt.xlabel("Principal Component 1")
        plt.ylabel("Principal Component 2")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig("pca_plot.png")
        plt.close()

        return df_ts, "pca_plot.png", f"Success: {len(df_ts)} samples predicted."

    except Exception as e:
        return None, None, f"Error: {str(e)}"
# --- UI Layout  ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# DIAPF: Dynamic Image Analysis & Processing Framework")
    with gr.Tabs():
        with gr.TabItem("1. Restoration & Enhancement"):
            with gr.Row():
                with gr.Column(scale=2):
                    src_img = gr.Image(label="Source Image", type="numpy", height=350, width=350)
                    mse_out = gr.Textbox(label="Metrics (MSE & PSNR)")
                with gr.Column(scale=3, variant="panel"):
                    gr.Markdown("### Phase 1 Control")
                    with gr.Accordion("Noise Injection", open=False):
                        n_t = gr.Dropdown(["None", "Salt & Pepper", "Gaussian", "Periodic", "Gamma Noise"], label="Noise Type", value="None"); n_l = gr.Slider(0, 0.2, 0.05, label="Intensity")
                    with gr.Accordion("Filters & Enhancement", open=True):
                        p_m = gr.Radio(["None", "Negative", "Log", "Gamma", "Histogram Equalization"], label="Transformation", value="None"); gam = gr.Slider(0.1, 5.0, 1.0, label="Gamma")
                        f_t = gr.Dropdown(["None", "Mean", "Median", "Laplacian", "Low Pass", "High Pass","Notch"], label="Filter", value="None"); k_s = gr.Slider(1, 15, 3, step=2, label="Kernel Size"); d_0 = gr.Slider(1, 200, 50, label="D0 Cutoff")
                with gr.Column(scale=2):
                    res_out = gr.Image(label="Restored", height=350, width=350); dif_out = gr.Image(label="Diff Map", height=200, width=350); his_out = gr.Image(label="Histogram", height=150, width=350)
            t1_in = [src_img, n_t, n_l, p_m, gam, f_t, k_s, d_0]; [i.change(dynamic_restoration, t1_in, [res_out, dif_out, his_out, mse_out]) for i in t1_in]

        with gr.TabItem("2. Segmentation & Morphology"):
            with gr.Row():
                with gr.Column(scale=1):
                    seg_src = gr.Image(label="Input Image", height=350, width=350)
                    gr.Markdown("### Phase 2: Structural Extraction")
                    sm = gr.Radio(["Global", "Adaptive","Otsu"], label="Method", value="Global"); st = gr.Slider(0, 255, 127, label="Threshold")
                    mo = gr.Dropdown(["None", "Erosion", "Dilation", "Opening", "Closing", "Boundary Extraction"], label="Operation", value="None")
                    sh = gr.Radio(["Square", "Cross", "Disk"], label="Shape", value="Square"); sz = gr.Slider(3, 15, 3, step=2, label="Size")
                with gr.Column(scale=1):
                    bin_out = gr.Image(label="Binary Mask", height=350, width=350); feat_out = gr.DataFrame(label="Features Vector", interactive=True)
                    gr.Markdown("> **IMPORTANT NOTE:** Use the 'Class Name' field below *only* for Training References. Leave it blank for Test Samples.")
                    c_lbl = gr.Textbox(label="Class Name (Required for Training Only)", placeholder="e.g. Coin_A", interactive=True)
                    file_out = gr.File(label="Download CSV Record")
            t2_in = [seg_src, sm, st, mo, sh, sz, c_lbl]; [i.change(dynamic_segmentation, t2_in, [bin_out, feat_out, file_out]) for i in t2_in]

        with gr.TabItem("3. Classification & PCA"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Phase 3: Analytical Analysis")
                    ftr = gr.File(label="Upload Training Reference (CSV)"); fts = gr.File(label="Upload Final Test Samples (CSV)")
                    bc = gr.Button("Run Analysis", variant="primary")
                with gr.Column():
                    pp = gr.Image(label="PCA Mapping"); cr = gr.Textbox(label="Status Report")
            ct = gr.DataFrame(label="Final Results Table"); bc.click(run_classification_analysis, [ftr, fts], [ct, pp, cr])

demo.launch()

/tmp/ipykernel_2306/481898864.py:311: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3a5c7ba500bf30426c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
pip install gradio
